Task 1: Import Libraries and Configure the **Model**

In [ ]:
import os
import glob
import zipfile
import urllib.request
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from google.colab import files
from IPython.display import display

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 3
SAMPLES_PER_CLASS = 1000

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

Task 2: Download and Extract the **Dataset**

In [ ]:
URL = "https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip"
ZIP_PATH = "/content/cell_images.zip"
DATA_PATH = "/content/malaria_data"

if not os.path.exists(ZIP_PATH):
    print("Downloading dataset (first run only)...")
    urllib.request.urlretrieve(URL, ZIP_PATH)

print("Extracting dataset...")
os.makedirs(DATA_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_PATH)

print("Dataset ready!")
print("Extracted folders:", os.listdir(DATA_PATH))

Task 3: Prepare and Split the **Dataset**

In [ ]:
CLASS_NAMES = ["Uninfected", "Parasitized"]

all_paths = []
all_labels = []

for label, name in enumerate(CLASS_NAMES):
    folders = glob.glob(
        os.path.join(DATA_PATH, "**", name),
        recursive=True
    )

    if not folders:
        raise FileNotFoundError(
            f"Cannot find the {name} folder. "
            "Check whether Task 3 extracted the dataset correctly."
        )

    image_files = []
    for ext in ("*.png", "*.jpg", "*.jpeg"):
        image_files.extend(
            glob.glob(os.path.join(folders[0], ext))
        )

    rng = np.random.default_rng(SEED)
    count = min(SAMPLES_PER_CLASS, len(image_files))
    chosen = rng.choice(image_files, count, replace=False)

    all_paths.extend(chosen.tolist())
    all_labels.extend([label] * count)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int32)

train_paths, temp_paths, train_y, temp_y = train_test_split(
    all_paths, all_labels,
    test_size=0.30, random_state=SEED,
    stratify=all_labels
)

val_paths, test_paths, val_y, test_y = train_test_split(
    temp_paths, temp_y,
    test_size=0.50, random_state=SEED,
    stratify=temp_y
)

print("Total images:", len(all_paths))
print("Training:", len(train_paths))
print("Validation:", len(val_paths))
print("Testing:", len(test_paths))
print("Labels: 0 = Uninfected, 1 = Parasitized")

Task 4: Image Preprocessing

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.float32)


def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            len(paths), seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_paths, train_y, training=True)
val_ds = make_dataset(val_paths, val_y)
test_ds = make_dataset(test_paths, test_y)

print("Image preprocessing complete.")

Task 5: Build the EfficientNetB0 Model Using Transfer Learning

In [ ]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_tensor=inputs
)

base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Task 6: Train the MicroScan AI **Model**

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=1,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

model.save("/content/microscan_ai.keras")
print("Training complete. Model saved.")

Task 7: Evaluate Model **Performance**

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print(f"\nTest accuracy: {test_accuracy * 100:.2f}%")
print(f"Test loss: {test_loss:.4f}")

test_probabilities = model.predict(test_ds, verbose=1).ravel()
test_predictions = (test_probabilities >= 0.5).astype(int)

print("\nClassification report:")
print(classification_report(
    test_y,
    test_predictions,
    labels=[0, 1],
    target_names=CLASS_NAMES,
    zero_division=0
))

cm = confusion_matrix(test_y, test_predictions, labels=[0, 1])

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title("Test Confusion Matrix")
plt.xticks([0, 1], CLASS_NAMES, rotation=15)
plt.yticks([0, 1], CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

Task 8: Single-Image **Prediction**

In [ ]:
def predict_image(image):
    image = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.asarray(image, dtype=np.float32)
    batch = np.expand_dims(arr, axis=0)

    probability = float(model.predict(batch, verbose=0)[0][0])

    if probability >= 0.5:
        label = "PARASITIZED"
        confidence = probability
    else:
        label = "UNINFECTED"
        confidence = 1 - probability

    return label, confidence, probability, arr


uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload an image.")

filename = next(iter(uploaded))

with Image.open(filename) as im:
    original = im.convert("RGB")

label, confidence, probability, arr = predict_image(original)

print("=" * 40)
print("          MICROSCAN AI")
print("=" * 40)
print("Prediction:", label)
print(f"Confidence: {confidence * 100:.2f}%")
print(f"Parasitized probability: {probability * 100:.2f}%")
print(f"Uninfected probability: {(1-probability) * 100:.2f}%")
print("=" * 40)

plt.figure(figsize=(5, 5))
plt.imshow(original)
plt.title("Uploaded Microscopic Cell Image")
plt.axis("off")
plt.show()

print("Research demonstration only—not a medical diagnosis.")

Task 9: Grad-CAM **Visualization**

In [ ]:
grad_model = Model(
    inputs=model.inputs,
    outputs=[
        base_model.get_layer("top_conv").output,
        model.output
    ]
)


def make_gradcam(image_array):
    batch = np.expand_dims(image_array, axis=0)

    with tf.GradientTape() as tape:
        conv_output, prediction = grad_model(batch, training=False)
        score = prediction[:, 0]

    gradients = tape.gradient(score, conv_output)

    if gradients is None:
        raise RuntimeError("Could not calculate Grad-CAM gradients.")

    weights = tf.reduce_mean(gradients, axis=(1, 2))
    heatmap = tf.reduce_sum(
        conv_output[0] * weights[0], axis=-1
    )
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    heatmap = tf.image.resize(
        heatmap[..., tf.newaxis],
        (IMG_SIZE, IMG_SIZE)
    ).numpy().squeeze()

    return heatmap


heatmap = make_gradcam(arr)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(arr.astype(np.uint8))
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(arr.astype(np.uint8))
plt.imshow(heatmap, cmap="jet", alpha=0.4)
plt.title("Grad-CAM Explanation")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Model location: /content/microscan_ai.keras")

Task 10: Save and Download the Trained **Model**

In [ ]:
files.download("/content/microscan_ai.keras")